# 05 Final Load Prep

This notebook prepares the final dataset for downstream use:
1. Feature engineering — Month, Day of Week, Price Bucket
2. Column renaming to snake_case
3. Drop is_price_outlier column
4. Final validation checks
5. Save final dataset to `data/processed/swiggy_final.csv`

In [1]:
from pathlib import Path

import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name.strip() == 'notebooks' else current_dir

In [2]:
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/swiggy_cleaned.csv'
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Order Date'])
print('Loaded:', df.shape)
df.head()

Loaded: (91788, 11)


,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,is_price_outlier
0,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25.0,False
1,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48.0,False
2,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34.0,False
3,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29.0,False
4,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51.0,False


## 1. Feature Engineering

In [3]:
df['Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['Day of Week'] = df['Order Date'].dt.day_name()
print('Month and Day of Week columns added.')

Month and Day of Week columns added.


In [4]:
bins   = [0, 150, 400, float('inf')]
labels = ['Budget', 'Mid-range', 'Premium']
df['Price Bucket'] = pd.cut(df['Price (INR)'], bins=bins, labels=labels, right=False)
print('Price Bucket distribution:')
print(df['Price Bucket'].value_counts())

Price Bucket distribution:
Price Bucket
Mid-range    54246
Budget       26915
Premium      10627
Name: count, dtype: int64


## 2. Drop is_price_outlier Column

In [5]:
df = df.drop(columns=['is_price_outlier'])
print('is_price_outlier column dropped.')

is_price_outlier column dropped.


## 3. Rename Columns to snake_case

In [6]:
rename_map = {
    'State'           : 'state',
    'City'            : 'city',
    'Order Date'      : 'order_date',
    'Restaurant Name' : 'restaurant_name',
    'Location'        : 'location',
    'Category'        : 'category',
    'Dish Name'       : 'dish_name',
    'Price (INR)'     : 'price_inr',
    'Rating'          : 'rating',
    'Rating Count'    : 'rating_count',
    'Month'           : 'month',
    'Day of Week'     : 'day_of_week',
    'Price Bucket'    : 'price_bucket'
}
df = df.rename(columns=rename_map)
print('Columns renamed to snake_case.')
print('Final columns:', df.columns.tolist())

Columns renamed to snake_case.
Final columns: ['state', 'city', 'order_date', 'restaurant_name', 'location', 'category', 'dish_name', 'price_inr', 'rating', 'rating_count', 'month', 'day_of_week', 'price_bucket']


## 4. Final Validation Checks

In [7]:
print('===== Final Validation =====')

null_count = df.isnull().sum().sum()
print(f'Null values      : {null_count} {"✓" if null_count == 0 else "✗ — check nulls"}')

dup_count = df.duplicated().sum()
print(f'Duplicate rows   : {dup_count} {"✓" if dup_count == 0 else "✗ — check duplicates"}')

print(f'Final row count  : {len(df):,}')
print(f'Final col count  : {len(df.columns)}')
print(f'Columns          : {df.columns.tolist()}')

===== Final Validation =====
Null values      : 0 ✓
Duplicate rows   : 0 ✓
Final row count  : 91,788
Final col count  : 13
Columns          : ['state', 'city', 'order_date', 'restaurant_name', 'location', 'category', 'dish_name', 'price_inr', 'rating', 'rating_count', 'month', 'day_of_week', 'price_bucket']


## 5. Save Final Dataset

In [9]:
FINAL_PATH = PROJECT_ROOT / 'data/processed/swiggy_final_cleaned.csv'
df.to_csv(FINAL_PATH, index=False)
print(f'Final dataset saved to: {FINAL_PATH}')
df.head()

Final dataset saved to: /Users/shreyashgolhani/Desktop/SectionD_G4_swiggy/data/processed/swiggy_final_cleaned.csv


,state,city,order_date,restaurant_name,location,category,dish_name,price_inr,rating,rating_count,month,day_of_week,price_bucket
0,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25.0,2025-04,Thursday,Budget
1,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48.0,2025-01,Wednesday,Budget
2,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34.0,2025-01,Tuesday,Budget
3,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29.0,2025-05,Friday,Mid-range
4,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51.0,2025-07,Wednesday,Mid-range
